# Exploratory Data Analysis

Аналіз реальних відгуків з Doc.ua для пошуку потенційних фейків unsupervised методами

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## 1. Завантаження даних

In [ ]:
# Завантажуємо датасет
df = pd.read_csv('../../data/doctors_reviews.csv')

print(f"Загальна кількість записів: {len(df):,}")
print(f"\nРозмірність датасету: {df.shape}")
print(f"\nКолонки: {df.columns.tolist()}")

In [ ]:

df.head(10)

In [ ]:

df.info()

## 2. Аналіз порожніх і заповнених відгуків

In [ ]:

missing_stats = pd.DataFrame({
    'Пропуски': df.isnull().sum(),
    'Відсоток': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing_stats)


empty_comments = df['Коментар'].isna().sum()
print(f"\nПорожніх коментарів: {empty_comments:,} ({empty_comments/len(df)*100:.2f}%)")

In [ ]:
# Візуалізація порожніх vs заповнених
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
empty_filled = df['Коментар'].isna().value_counts()
ax[0].pie(empty_filled, labels=['Заповнені', 'Порожні'], autopct='%1.1f%%', startangle=90)
ax[0].set_title('Розподіл порожніх та заповнених відгуків')

# бар чарт для інших колонок
missing_data = df.isnull().sum()
missing_data.plot(kind='bar', ax=ax[1], color='coral')
ax[1].set_title('Кількість пропусків по колонках')
ax[1].set_ylabel('Кількість пропусків')
ax[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Аналіз довжини відгуків

In [ ]:
# Створимо копію з непорожніми коментарями
df_with_comments = df[df['Коментар'].notna()].copy()

# Додаємо статистики довжини
df_with_comments['text_length'] = df_with_comments['Коментар'].str.len()
df_with_comments['word_count'] = df_with_comments['Коментар'].str.split().str.len()

print(f"Відгуків з текстом: {len(df_with_comments):,}")
print(f"\nСтатистика довжини тексту (символів):")
print(df_with_comments['text_length'].describe())
print(f"\nСтатистика кількості слів:")
print(df_with_comments['word_count'].describe())

In [ ]:
# Візуалізація розподілу довжини
fig, ax = plt.subplots(2, 2, figsize=(15, 10))

# гыстограмма довжини тексту
ax[0, 0].hist(df_with_comments['text_length'], bins=50, edgecolor='black', alpha=0.7)
ax[0, 0].set_xlabel('Довжина тексту (символів)')
ax[0, 0].set_ylabel('Кількість відгуків')
ax[0, 0].set_title('Розподіл довжини відгуків')
ax[0, 0].axvline(df_with_comments['text_length'].median(), color='red', linestyle='--', label=f'Медіана: {df_with_comments["text_length"].median():.0f}')
ax[0, 0].legend()

# гістограмма кількості слів
ax[0, 1].hist(df_with_comments['word_count'], bins=50, edgecolor='black', alpha=0.7, color='green')
ax[0, 1].set_xlabel('Кількість слів')
ax[0, 1].set_ylabel('Кількість відгуків')
ax[0, 1].set_title('Розподіл кількості слів')
ax[0, 1].axvline(df_with_comments['word_count'].median(), color='red', linestyle='--', label=f'Медіана: {df_with_comments["word_count"].median():.0f}')
ax[0, 1].legend()

# Boxplot довжини
ax[1, 0].boxplot(df_with_comments['text_length'], vert=False)
ax[1, 0].set_xlabel('Довжина тексту (символів)')
ax[1, 0].set_title('Boxplot довжини відгуків')

# Boxplot кількості слів
ax[1, 1].boxplot(df_with_comments['word_count'], vert=False)
ax[1, 1].set_xlabel('Кількість слів')
ax[1, 1].set_title('Boxplot кількості слів')

plt.tight_layout()
plt.show()

## 4. Аналіз дуже коротких відгуків

In [ ]:
# Дуже короткі відгуки (< 30 символів)
very_short = df_with_comments[df_with_comments['text_length'] < 30]
print(f"Дуже коротких відгуків (< 30 символів): {len(very_short):,} ({len(very_short)/len(df_with_comments)*100:.2f}%)")
print(f"\nПриклади дуже коротких відгуків:")
print(very_short['Коментар'].head(20).tolist())

## 5. Аналіз лікарів та міст

In [ ]:
# Унікальні міста та лікарі
doctor_name_col_name = "Ім'я лікаря"

print(f"Унікальних міст: {df['Місто'].nunique()}")
print(f"Унікальних лікарів: {df[doctor_name_col_name].nunique()}")
print(f"\nТоп-10 міст за кількістю відгуків:")
print(df['Місто'].value_counts().head(10))

In [ ]:
# Топ лікарів за кількістю відгуків
top_doctors = df["Ім'я лікаря"].value_counts().head(20)
print("Топ-20 лікарів за кількістю відгуків:")
print(top_doctors)

In [ ]:
# Візуалізація топ лікарів
fig, ax = plt.subplots(figsize=(12, 8))
top_doctors.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Кількість відгуків')
ax.set_ylabel('Лікар')
ax.set_title('Топ-20 лікарів за кількістю відгуків')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Temporal Analysis - Розподіл відгуків по датах

In [ ]:
# Парсинг дат (eg. "10 Жовтень 2025")
month_mapping = {
    'Січень': 1, 'Лютий': 2, 'Березень': 3, 'Квітень': 4,
    'Травень': 5, 'Червень': 6, 'Липень': 7, 'Серпень': 8,
    'Вересень': 9, 'Жовтень': 10, 'Листопад': 11, 'Грудень': 12
}

def parse_ukrainian_date(date_str):
    if pd.isna(date_str):
        return None
    try:
        parts = date_str.strip().split()
        if len(parts) == 3:
            day, month_ukr, year = parts
            month = month_mapping.get(month_ukr)
            if month:
                return datetime(int(year), month, int(day))
    except:
        pass
    return None

df['parsed_date'] = df['Дата коментаря'].apply(parse_ukrainian_date)
print(f"Успішно розпарсено дат: {df['parsed_date'].notna().sum():,} з {len(df):,}")

In [ ]:
# Статистика по датах
df_with_dates = df[df['parsed_date'].notna()].copy()
print(f"Найстаріший відгук: {df_with_dates['parsed_date'].min()}")
print(f"Найновіший відгук: {df_with_dates['parsed_date'].max()}")
print(f"Часовий діапазон: {(df_with_dates['parsed_date'].max() - df_with_dates['parsed_date'].min()).days} днів")

In [ ]:
# Розподіл по місяцях
df_with_dates['year_month'] = df_with_dates['parsed_date'].dt.to_period('M')
monthly_counts = df_with_dates['year_month'].value_counts().sort_index()

plt.figure(figsize=(14, 6))
monthly_counts.plot(kind='line', marker='o')
plt.xlabel('Місяць')
plt.ylabel('Кількість відгуків')
plt.title('Розподіл відгуків по місяцях')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Review Bursts - Пошук аномальних спалахів відгуків

In [ ]:
# Групуємо відгуки по лікарю та даті
doctor_daily_reviews = df_with_dates.groupby(['Ім\'я лікаря', df_with_dates['parsed_date'].dt.date]).size().reset_index(name='reviews_per_day')

# Топ днів з найбільшою кількістю відгуків для одного лікаря
top_bursts = doctor_daily_reviews.nlargest(20, 'reviews_per_day')
print("Топ-20 днів з найбільшою кількістю відгуків для одного лікаря:")
print(top_bursts)

In [ ]:
# Статистика bursts
print(f"\nСтатистика відгуків на день для одного лікаря:")
print(doctor_daily_reviews['reviews_per_day'].describe())

# Візуалізація розподілу
plt.figure(figsize=(12, 6))
plt.hist(doctor_daily_reviews['reviews_per_day'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Кількість відгуків на день для одного лікаря')
plt.ylabel('Частота')
plt.title('Розподіл щоденних відгуків по лікарях')
plt.axvline(doctor_daily_reviews['reviews_per_day'].median(), color='red', linestyle='--', label=f'Медіана: {doctor_daily_reviews["reviews_per_day"].median():.1f}')
plt.axvline(doctor_daily_reviews['reviews_per_day'].quantile(0.95), color='orange', linestyle='--', label=f'95-й перцентиль: {doctor_daily_reviews["reviews_per_day"].quantile(0.95):.1f}')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Аналіз анонімних відгуків

In [ ]:
# Аналіз коментаторів
anonymous_count = df[df["Ім'я коментатора"] == 'Анонім'].shape[0]
missing_name_count = df["Ім'я коментатора"].isna().sum()
named_count = df[(df["Ім'я коментатора"] != 'Анонім') & (df["Ім'я коментатора"].notna())].shape[0]

print(f"Анонімних: {anonymous_count:,} ({anonymous_count/len(df)*100:.2f}%)")
print(f"Без імені: {missing_name_count:,} ({missing_name_count/len(df)*100:.2f}%)")
print(f"З іменем: {named_count:,} ({named_count/len(df)*100:.2f}%)")

# Візуалізація
fig, ax = plt.subplots(figsize=(8, 8))
labels = ['Анонім', 'Без імені', 'З іменем']
sizes = [anonymous_count, missing_name_count, named_count]
colors = ['#ff9999', '#ffcc99', '#99ff99']
ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('Розподіл відгуків за типом коментатора')
plt.show()

In [ ]:
# Зберігаємо оброблений датасет для наступних експериментів
df.to_csv('../../data/doctors_reviews_with_features.csv', index=False)
print("Датасет з додатковими фічами збережено!")